# Agentic AI Fundamentals

## Notebook 04: Real Gemini API and Native Function Calling

This notebook replaces the scripted model decisions from earlier chapters with a real Gemini API request. Gemini will receive the user's message and a collection of tool schemas. It may return a normal answer or a structured function call containing a tool name and arguments.

We will handle the function call manually rather than enabling automatic execution. This makes the trust boundary visible: Gemini proposes a function call, our registry validates and executes it, and the resulting observation is sent back to Gemini for the final response.

The notebook is fully self-contained for Google Colab. It does not import any local project files.

## What this notebook implements

The complete interaction contains two API turns:

                    ┌─────────────────────────────┐
                    │ User request + tool schemas │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │ Gemini returns a            │
                    │ function call               │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │ Python validates and        │
                    │ executes the tool           │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │ Python sends the function   │
                    │ result to Gemini            │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │ Gemini produces a           │
                    │ user-facing answer          │
                    └─────────────────────────────┘

This is one tool-calling cycle, not yet a general agent loop. The next notebook will repeat the cycle until the model returns a final answer or a safety limit is reached.

## 1. Install the notebook dependencies

Colab runtimes are temporary, so installation belongs inside the notebook. Run the next cell once after opening a fresh runtime. If Colab asks you to restart the session after an upgrade, restart it and continue from the following cell.

In [ ]:
%pip install -q -U "google-genai>=1.0.0" "pydantic>=2.7,<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## 2. Store the API key as a Colab secret

Open the key icon in the left sidebar, create a secret named `GEMINI_API_KEY`, paste your key, and enable notebook access. The key should never be written directly into a notebook that may be committed to GitHub.

The helper below first checks Colab Secrets and then falls back to the `GEMINI_API_KEY` environment variable. This keeps the notebook usable outside Colab without changing its main code.

In [ ]:
import os


def load_gemini_api_key() -> str:
    try:
        from google.colab import userdata

        api_key = userdata.get("GEMINI_API_KEY")
    except (ImportError, KeyError):
        api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY was not found. Add it to Colab Secrets or "
            "set it as an environment variable."
        )

    return api_key


GEMINI_API_KEY = load_gemini_api_key()
print("API key loaded successfully.")

API key loaded successfully.


## 3. Imports and model configuration

The `google-genai` package is Google's current Python SDK for the Gemini API. We use the Interactions API because it exposes the function-call and function-result steps clearly.

Model availability can change over time. The model below matches the current official function-calling examples when this tutorial was written. If it is unavailable in your account or region, replace it with a current Gemini model that supports function calling.

In [ ]:
import json
import re
from dataclasses import dataclass
from typing import Any, Callable

from pydantic import BaseModel, ConfigDict, Field, ValidationError


MODEL_NAME = "gemini-3.8-flash"
print("Configured model:", MODEL_NAME)

Configured model: gemini-3.8-flash


In [ ]:
from google import genai


client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 4. Define local capabilities

We provide two read-only tools. `multiply` handles exact arithmetic, while `search_course_notes` retrieves information from a small local knowledge base. Gemini receives descriptions of these functions but does not receive direct access to the Python objects.

In [ ]:
def multiply(first_number: float, second_number: float) -> float:
    return first_number * second_number


COURSE_NOTES = {
    "agent": (
        "An agent uses a model-driven decision inside a controlled loop, "
        "with tools, state, and explicit stop conditions."
    ),
    "tool": (
        "A tool is a capability exposed by application code through a "
        "validated and governed interface."
    ),
    "observation": (
        "A tool result becomes an observation that can influence the "
        "model's next decision."
    ),
}


def search_course_notes(query: str, max_results: int = 2) -> list[dict[str, str]]:
    query_tokens = set(re.findall(r"[a-z0-9]+", query.casefold()))
    scored_notes = []

    for topic, text in COURSE_NOTES.items():
        note_tokens = set(re.findall(r"[a-z0-9]+", f"{topic} {text}".casefold()))
        score = len(query_tokens & note_tokens)

        if score:
            scored_notes.append((score, topic, text))

    scored_notes.sort(reverse=True)

    return [
        {"topic": topic, "text": text}
        for _, topic, text in scored_notes[:max_results]
    ]

## 5. Define argument contracts

Pydantic remains our local validation layer even though the provider also receives a JSON schema. Provider-side structured generation improves reliability, while local validation protects the execution boundary.

In [ ]:
class MultiplyArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    first_number: float = Field(description="The first multiplication operand")
    second_number: float = Field(description="The second multiplication operand")


class SearchCourseNotesArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query: str = Field(
        min_length=3,
        max_length=300,
        description="A focused query for the local Agentic AI course notes",
    )
    max_results: int = Field(default=2, ge=1, le=3)

## 6. Recreate the tool and registry layer inside the notebook

Because this course targets Colab, all required classes are defined here rather than imported from external Python modules. Later notebooks may repeat this foundation before adding new behavior; that repetition keeps every chapter independently runnable.

In [ ]:
@dataclass(frozen=True)
class Tool:
    name: str
    description: str
    arguments_model: type[BaseModel]
    function: Callable[..., Any]

    def execute(self, raw_arguments: dict[str, Any]) -> Any:
        validated = self.arguments_model.model_validate(raw_arguments)
        return self.function(**validated.model_dump())

    def as_gemini_schema(self) -> dict[str, Any]:
        return {
            "type": "function",
            "name": self.name,
            "description": self.description,
            "parameters": self.arguments_model.model_json_schema(),
        }


class ToolRegistry:
    def __init__(self, tools: list[Tool] | None = None) -> None:
        self._tools: dict[str, Tool] = {}

        for tool in tools or []:
            self.register(tool)

    def register(self, tool: Tool) -> None:
        if tool.name in self._tools:
            raise ValueError(f"Duplicate tool name: {tool.name}")

        self._tools[tool.name] = tool

    def schemas(self) -> list[dict[str, Any]]:
        return [tool.as_gemini_schema() for tool in self._tools.values()]

    def execute(self, tool_name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        tool = self._tools.get(tool_name)

        if tool is None:
            return {
                "success": False,
                "error_type": "unknown_tool",
                "error": f"Tool is not registered: {tool_name}",
            }

        try:
            result = tool.execute(arguments)
        except ValidationError as error:
            return {
                "success": False,
                "error_type": "invalid_arguments",
                "error": error.errors(),
            }
        except Exception as error:
            return {
                "success": False,
                "error_type": "execution_error",
                "error": f"{type(error).__name__}: {error}",
            }

        return {"success": True, "result": result}

In [ ]:
registry = ToolRegistry(
    [
        Tool(
            name="multiply",
            description=(
                "Multiply two numbers when an exact multiplication result "
                "is required. Do not use for other arithmetic operations."
            ),
            arguments_model=MultiplyArguments,
            function=multiply,
        ),
        Tool(
            name="search_course_notes",
            description=(
                "Search the local Agentic AI course notes for definitions "
                "that are not already available in the conversation."
            ),
            arguments_model=SearchCourseNotesArguments,
            function=search_course_notes,
        ),
    ]
)

gemini_tools = registry.schemas()
print(json.dumps(gemini_tools, indent=2))

[
  {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers when an exact multiplication result is required. Do not use for other arithmetic operations.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "first_number": {
          "description": "The first multiplication operand",
          "title": "First Number",
          "type": "number"
        },
        "second_number": {
          "description": "The second multiplication operand",
          "title": "Second Number",
          "type": "number"
        }
      },
      "required": [
        "first_number",
        "second_number"
      ],
      "title": "MultiplyArguments",
      "type": "object"
    }
  },
  {
    "type": "function",
    "name": "search_course_notes",
    "description": "Search the local Agentic AI course notes for definitions that are not already available in the conversation.",
    "parameters": {
      "additionalProperties": false,
  

## 7. Make the first API request

The user prompt requires exact arithmetic, so we expect Gemini to request `multiply`. The `tools` parameter advertises the available functions; it does not execute them.

Model behavior is not perfectly deterministic, so application code must support both possible outcomes: a function call or a direct textual answer.

In [ ]:
user_prompt = (
    "What is 37 multiplied by 46? Use an available tool when it provides "
    "a more reliable result."
)

first_interaction = client.interactions.create(
    model=MODEL_NAME,
    input=user_prompt,
    tools=gemini_tools,
)

print("Interaction ID:", first_interaction.id)
print("Direct output   :", first_interaction.output_text)
print("Step types      :", [step.type for step in first_interaction.steps])

Interaction ID: v1_ChdwSHV6YXVHS0U3YTJfUFVQdzVQR21BYxIXcEh1emF1R0tFN2EyX1BVUHc1UEdtQWM
Direct output   : 
Step types      : ['thought', 'function_call']


## 8. Inspect the model's function call

The Interactions API returns a sequence of typed steps. We select the function-call step explicitly rather than assuming it is always the first step.

In [ ]:
function_calls = [
    step
    for step in first_interaction.steps
    if step.type == "function_call"
]

if not function_calls:
    print("Gemini answered without requesting a function:")
    print(first_interaction.output_text)
else:
    for function_call in function_calls:
        print("Call ID   :", function_call.id)
        print("Tool name :", function_call.name)
        print("Arguments :", function_call.arguments)

Call ID   : call_531432
Tool name : multiply
Arguments : {'first_number': 37, 'second_number': 46}


Gemini has only proposed an action at this stage. The function has not run. We still control registration checks, argument validation, and execution.

## 9. Validate and execute the requested function

For this introductory cycle, we expect one function call. Parallel calls will be handled in the general loop in the next notebook.

In [ ]:
if not function_calls:
    raise RuntimeError(
        "No function call was returned. Try the request again or use a "
        "prompt that clearly requires exact multiplication."
    )

function_call = function_calls[0]

tool_observation = registry.execute(
    tool_name=function_call.name,
    arguments=dict(function_call.arguments),
)

print("Validated observation:")
print(tool_observation)

Validated observation:
{'success': True, 'result': 1702.0}


If Gemini produced a misspelled argument or unsupported tool name, the registry would return a structured failure. We would send that observation back to the model rather than silently modifying the request.

## 10. Return the function result to Gemini

The second request references the previous interaction and supplies a `function_result` with the original call ID. This allows Gemini to connect the observation to its earlier request and produce a natural-language answer.

In [ ]:
final_interaction = client.interactions.create(
    model=MODEL_NAME,
    previous_interaction_id=first_interaction.id,
    tools=gemini_tools,
    input=[
        {
            "type": "function_result",
            "name": function_call.name,
            "call_id": function_call.id,
            "result": [
                {
                    "type": "text",
                    "text": json.dumps(tool_observation),
                }
            ],
        }
    ],
)

print(final_interaction.output_text)

37 multiplied by 46 is **1,702**.


A successful response should state that `37 × 46 = 1702`, although its wording may differ. The numeric result originated from our local Python function; Gemini converted the observation into a user-facing answer.

## 11. Try the retrieval tool

The same tool-calling protocol works for knowledge augmentation. Run the following cells after the arithmetic example to start a separate interaction.

In [ ]:
retrieval_prompt = (
    "According to the local course notes, what is an observation in an "
    "agent system? Use the course-notes tool."
)

retrieval_interaction = client.interactions.create(
    model=MODEL_NAME,
    input=retrieval_prompt,
    tools=gemini_tools,
)

retrieval_calls = [
    step
    for step in retrieval_interaction.steps
    if step.type == "function_call"
]

if not retrieval_calls:
    raise RuntimeError("Gemini did not request the retrieval tool")

retrieval_call = retrieval_calls[0]
retrieval_observation = registry.execute(
    retrieval_call.name,
    dict(retrieval_call.arguments),
)

print(retrieval_call.name, retrieval_call.arguments)
print(retrieval_observation)

search_course_notes {'query': 'observation agent system'}
{'success': True, 'result': [{'topic': 'observation', 'text': "A tool result becomes an observation that can influence the model's next decision."}, {'topic': 'agent', 'text': 'An agent uses a model-driven decision inside a controlled loop, with tools, state, and explicit stop conditions.'}]}


In [ ]:
retrieval_final = client.interactions.create(
    model=MODEL_NAME,
    previous_interaction_id=retrieval_interaction.id,
    tools=gemini_tools,
    input=[
        {
            "type": "function_result",
            "name": retrieval_call.name,
            "call_id": retrieval_call.id,
            "result": [
                {
                    "type": "text",
                    "text": json.dumps(retrieval_observation),
                }
            ],
        }
    ],
)

print(retrieval_final.output_text)

According to the local course notes, an observation is a tool result that can influence the model's next decision within an agent system.


## 12. What the API did—and did not do

Gemini performed two model responsibilities:

- it selected a tool and generated structured arguments;
- after receiving the observation, it formulated the final answer.

Our application retained the operational responsibilities:

- it defined the available action space;
- it validated the generated arguments;
- it executed the local function;
- it decided what result to return; and
- it connected the result to the correct call ID.

Function calling is therefore a protocol between a model and an application, not remote execution performed by the model.

## 13. Common Colab and API problems

| Symptom | Likely cause | Suggested check |
|---|---|---|
| `GEMINI_API_KEY was not found` | Secret is missing or notebook access is disabled | Check the key icon in the Colab sidebar |
| Authentication or permission error | Invalid key or unavailable API | Create or verify the key in Google AI Studio |
| Model-not-found error | Model availability changed | Choose a current function-calling Gemini model |
| No function call returned | The model chose to answer directly | Inspect `output_text` or make the tool need clearer |
| Argument validation error | Generated arguments did not satisfy Pydantic | Inspect the structured observation; do not bypass validation |
| Rate-limit error | Request quota was reached | Wait and retry with bounded backoff |

Avoid placing an API call inside an unbounded retry loop. Retries should be limited and reserved for transient failures.

## 14. Exercise: inspect direct-answer behavior

Send the prompt `"Explain in one sentence why a tool registry is useful."` with the same `gemini_tools`. Do not instruct the model to use a tool.

Inspect both `output_text` and the step types. Did Gemini answer directly, or did it search the local notes? Neither path is automatically wrong; the purpose is to observe that `auto` tool choice allows the model to decide.

Then change the prompt to explicitly request the local course notes and compare the trajectory.

In [ ]:
# Write and run the two comparison requests here.

## Notebook summary

We connected a real Gemini model to two local Python tools through native function calling. Gemini selected a function and generated arguments, while our notebook validated and executed the request through a controlled registry. We then returned the observation with its call ID and received a final user-facing answer.

The next notebook will turn this two-turn example into a reusable agent loop. It will support direct answers, multiple sequential tool calls, parallel calls, structured traces, maximum-step limits, and controlled error recovery.

### References

- [Gemini API function calling](https://ai.google.dev/gemini-api/docs/function-calling)
- [Google Gen AI Python SDK](https://googleapis.github.io/python-genai/)